<a href="https://colab.research.google.com/github/RajeshworM/IMD_GRID_DATA_EXTRACTION/blob/master/Weather_Transform_Varified.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<font color ='blue'> <font size ='5'>**Climate data Transformation Daily to Weekly Data**

In [3]:
# =========================================================
# FULL AUTOMATED DISTRICT WEATHER PROCESSING PIPELINE
# WITH VERIFICATION FOR RAINY DAYS
# =========================================================

import pandas as pd
import numpy as np
from google.colab import files

# =========================================================
# STEP 1: State Code Map
# =========================================================
STATE_MAP = {
    'HR': 'Haryana',
    'UP': 'Uttar Pradesh',
    'PB': 'Punjab',
    'HP': 'Himachal Pradesh',
    'JK': 'Jammu & Kashmir',
    'UT': 'Uttarakhand'
}

# =========================================================
# STEP 2: Upload ALL CSV Files
# =========================================================
uploaded = files.upload()
dfs = []

for file_name in uploaded.keys():
    if not file_name.endswith('.csv'):
        continue

    parts = file_name.split('_')
    state_code = parts[0]
    district_code = parts[1]

    state = STATE_MAP.get(state_code, state_code)
    district = district_code

    temp_df = pd.read_csv(file_name)
    temp_df['State'] = state
    temp_df['District'] = district

    dfs.append(temp_df)

df = pd.concat(dfs, ignore_index=True)

print("✅ Files read:", len(dfs))
print("✅ Rows before cleaning:", df.shape[0])

# =========================================================
# STEP 3: Robust Date Parsing
# =========================================================
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
df = df.dropna(subset=['Date'])

# =========================================================
# STEP 4: CLEAN ALL NUMERIC VARIABLES
# =========================================================
CLEAN_RULES = {
    'Rainfall (mm)':        (0, 500),
    'Max_Temperature (C)':  (0, 60),
    'Min_Temperature (C)':  (-10, 50),
    'Relative_Humidity (%)':(0, 100),
    'Wind_Speed (m/s)':     (0, 35),
    'Sunshine_Hours (hrs)': (0, 15),
    'Solar Radiation (mm/day)': (0, 30)
}

for col, (low, high) in CLEAN_RULES.items():
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df.loc[(df[col] < low) | (df[col] > high), col] = np.nan

print("✅ Numeric cleaning applied to all variables")

# =========================================================
# STEP 5: Block → District (Daily Aggregation)
# =========================================================
district_daily = (
    df.groupby(['State', 'District', 'Date'])
      .agg({
          'Rainfall (mm)': 'mean',
          'Max_Temperature (C)': 'mean',
          'Min_Temperature (C)': 'mean',
          'Relative_Humidity (%)': 'mean',
          'Wind_Speed (m/s)': 'mean',
          'Sunshine_Hours (hrs)': 'mean',
          'Solar Radiation (mm/day)': 'mean'
      })
      .reset_index()
)

# =========================================================
# STEP 6: Derived Daily Indicators (SAFE)
# =========================================================
district_daily['Tmean'] = district_daily[[
    'Max_Temperature (C)', 'Min_Temperature (C)'
]].mean(axis=1, skipna=True)

district_daily['Rainy_Day'] = (district_daily['Rainfall (mm)'] >= 2.5).astype(int)
district_daily['No_Rain_Day'] = (district_daily['Rainfall (mm)'] < 2.5).astype(int)
district_daily['Excess_Rain_Day'] = (district_daily['Rainfall (mm)'] > 30).astype(int)

district_daily['Heat_Stress'] = (district_daily['Max_Temperature (C)'] > 35).astype(int)
district_daily['Cold_Stress'] = (district_daily['Min_Temperature (C)'] < 18).astype(int)

# =========================================================
# STEP 7: Year & ISO Week
# =========================================================
district_daily['Year'] = district_daily['Date'].dt.year
district_daily['Week'] = district_daily['Date'].dt.isocalendar().week.astype(int)

# =========================================================
# STEP 8: Daily → Weekly Aggregation
# =========================================================
district_weekly = (
    district_daily.groupby(['State', 'District', 'Year', 'Week'])
    .agg({
        'Rainfall (mm)': 'sum',
        'Rainy_Day': 'sum',
        'No_Rain_Day': 'sum',
        'Excess_Rain_Day': 'sum',
        'Max_Temperature (C)': 'mean',
        'Min_Temperature (C)': 'mean',
        'Tmean': 'mean',
        'Relative_Humidity (%)': 'mean',
        'Wind_Speed (m/s)': 'mean',
        'Sunshine_Hours (hrs)': 'sum',
        'Solar Radiation (mm/day)': 'sum',
        'Heat_Stress': 'sum',
        'Cold_Stress': 'sum'
    })
    .reset_index()
)

# =========================================================
# STEP 8b: CAP DAY COUNTS TO 7 AND ALIGN RainyDays + NoRainDays
# =========================================================
total_days = 7
day_cols = ['Rainy_Day', 'No_Rain_Day', 'Excess_Rain_Day', 'Heat_Stress', 'Cold_Stress']

for col in day_cols:
    district_weekly[col] = district_weekly[col].clip(upper=total_days)

# Align NoRainDays + RainyDays = total_days
district_weekly['No_Rain_Day'] = total_days - district_weekly['Rainy_Day']

district_weekly.rename(columns={
    'Rainfall (mm)': 'Rainfall',
    'Rainy_Day': 'RainyDays',
    'No_Rain_Day': 'NoRainDays',
    'Excess_Rain_Day': 'ExcessRainDays',
    'Max_Temperature (C)': 'Tmax',
    'Min_Temperature (C)': 'Tmin',
    'Relative_Humidity (%)': 'RH',
    'Wind_Speed (m/s)': 'Wind',
    'Sunshine_Hours (hrs)': 'Sunshine',
    'Solar Radiation (mm/day)': 'SolarRad'
}, inplace=True)

# =========================================================
# STEP 9: Enforce Weeks 1–53
# =========================================================
all_weeks = pd.DataFrame({'Week': range(1, 54)})

expanded = []
for (state, district, year), g in district_weekly.groupby(['State', 'District', 'Year']):
    g = all_weeks.merge(g, on='Week', how='left')
    g[['State', 'District', 'Year']] = [state, district, year]

    zero_safe = [
        'Rainfall', 'RainyDays', 'NoRainDays', 'ExcessRainDays',
        'Sunshine', 'SolarRad', 'Heat_Stress', 'Cold_Stress'
    ]
    g[zero_safe] = g[zero_safe].fillna(0)

    # Re-cap day counts
    for col in ['RainyDays', 'NoRainDays', 'ExcessRainDays', 'Heat_Stress', 'Cold_Stress']:
        g[col] = g[col].clip(upper=total_days)

    # Align RainyDays + NoRainDays = 7
    g['NoRainDays'] = total_days - g['RainyDays']

    expanded.append(g)

district_weekly_full = pd.concat(expanded, ignore_index=True)

# =========================================================
# STEP 10: Wide Format (53 Weeks)
# =========================================================
id_cols = ['State', 'District', 'Year']
value_cols = [c for c in district_weekly_full.columns if c not in id_cols + ['Week']]

wide_df = district_weekly_full.pivot_table(
    index=id_cols,
    columns='Week',
    values=value_cols
)

wide_df.columns = [f"{v}_W{w}" for v, w in wide_df.columns]
wide_df = wide_df.reset_index()

# =========================================================
# STEP 11: SAVE ONE FILE PER DISTRICT IN ZIP
import zipfile
import os

for state, state_df in wide_df.groupby('State'):

    zip_state_name = state.replace(" ", "_")
    zip_name = f"{zip_state_name}_District_Weekly_Weather_53W.zip"

    with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zipf:

        for (st, district), g in state_df.groupby(['State', 'District']):
            state_code = [k for k, v in STATE_MAP.items() if v == st]
            state_code = state_code[0] if state_code else st[:2].upper()

            output_file = f"{state_code}_{district}_district_weekly_weather_WIDE_53weeks.csv"
            g.to_csv(output_file, index=False)

            zipf.write(output_file)
            os.remove(output_file)   # cleanup temp CSV

    files.download(zip_name)
    print(f"📦 Saved & downloaded: {zip_name}")

print("✅ ALL DISTRICTS PROCESSED — FULLY CLEAN & VERIFIED")


Saving HR_AMB_Ambala-I.csv to HR_AMB_Ambala-I (2).csv
Saving HR_AMB_Ambala-II.csv to HR_AMB_Ambala-II (2).csv
Saving HR_AMB_Barara.csv to HR_AMB_Barara (2).csv
Saving HR_AMB_Naraingarh.csv to HR_AMB_Naraingarh (2).csv
Saving HR_AMB_Saha.csv to HR_AMB_Saha (2).csv
Saving HR_AMB_Shahzadpur.csv to HR_AMB_Shahzadpur (2).csv
Saving HR_BHW_Bawani Khera.csv to HR_BHW_Bawani Khera (2).csv
Saving HR_BHW_Behal.csv to HR_BHW_Behal (2).csv
✅ Files read: 8
✅ Rows before cleaning: 116880
✅ Numeric cleaning applied to all variables


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

📦 Saved & downloaded: Haryana_District_Weekly_Weather_53W.zip
✅ ALL DISTRICTS PROCESSED — FULLY CLEAN & VERIFIED
